# TNBC Drug Response Predictor — 02 Modeling + Killer Ranking (Colab)

**Input:** `/content/tnbc_ml_dataset.parquet` from `01_Data_Audit` (rows = cell_line × drug, features = `pca_0..127 + drug_one_hot`, targets = `auc`, `ln_ic50`).  
**Output (this notebook guarantees):**
1. A **regression model that predicts sensitivity** `P(pca_vector, drug) → predicted_AUC` (lower = more killing).
2. A **ranked killer list for TNBC** — Top-20 drugs for `MDA-MB-231` + selectivity vs `MCF7` control.

PRD map: FR-05 training order Dummy→RF→XGB→CatBoost, GroupKFold by cell_line (§109), FR-06 MAE/RMSE/R²/Spearman + NDCG@10/Recall@10 (AUC<0.3 sensitive), FR-07 SHAP, §10 wet-lab shortlist (Top 3-5 + Doxorubicin control).
**Notebook version: 2026-10-12h** — check §0 prints this version.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────
import sys, os, gc, json, time
print(sys.version)
!cat /proc/meminfo | head -2
try:
    import catboost, xgboost, shap  # noqa: F401
    print('ml libs present — skipping pip install')
except ImportError:
    print('installing ml libs (Colab/fresh env)...')
    get_ipython().system('pip -q install -U "xgboost==2.0.3" catboost shap pyarrow 2>&1 | tail -2')
import pandas as pd, numpy as np
print('pandas', pd.__version__)
NB_VERSION = '2026-10-12h (02-model)'; print('notebook version:', NB_VERSION)

In [ ]:
# ── 1. Load gold dataset + drug names (/content only is fine) ──────────
# Paths: Colab (/content) or LOCAL (./outputs_local) — auto-detected
import os
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
OUT = '/content' if ON_COLAB else os.path.abspath('outputs_local')
os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT, '| ON_COLAB =', ON_COLAB)


import os
CANDS = [OUT + '/tnbc_ml_dataset.parquet',
         OUT + '/drive/MyDrive/TNBC_tnbc_ml_dataset.parquet',
         OUT + '/drive/MyDrive/TNBC/data/gold/tnbc_ml_dataset.parquet']
ML_PATH = next((p for p in CANDS if os.path.exists(p)), None)
assert ML_PATH, 'MISSING tnbc_ml_dataset.parquet — run 01 notebook §10 first!'
print('ML dataset:', ML_PATH, f"({os.path.getsize(ML_PATH)/1e6:.1f} MB)")

ml = pd.read_parquet(ML_PATH)
# Defensive: drop duplicate (line, drug) rows from ANY 01 output (dup PCA rows inflate training)
_n0 = len(ml)
ml = ml.drop_duplicates(subset=['ModelID', 'drug_id'], keep='first').reset_index(drop=True)
print(f'deduped ML rows: {len(ml)} (dropped {_n0 - len(ml)} duplicate pairs)')
print(ml.shape)
display(ml.head(2))
print('cell lines:', ml['ModelID'].nunique(), '| drugs:', ml['drug_id'].nunique())
print('AUC range:', float(ml['auc'].min()), '→', float(ml['auc'].max()), '(must be [0,1] per PRD §7)')
assert float(ml['auc'].min()) >= 0 and float(ml['auc'].max()) <= 1

# Drug ID → human-readable name (for the killer list)
name_map = {}
for cand in [OUT + '/REPURPOSINGLog2ViabilityCollapsedConditions.csv',
              'Data/REPURPOSINGLog2ViabilityCollapsedConditions.csv',
             OUT + '/drive/MyDrive/TNBC/data/raw/REPURPOSINGLog2ViabilityCollapsedConditions.csv']:
    if os.path.exists(cand):
        cmap = pd.read_csv(cand, usecols=['CompoundName','CompoundID']).drop_duplicates('CompoundID')
        name_map = dict(zip(cmap['CompoundID'], cmap['CompoundName']))
        print(f'drug names from {cand}: {len(name_map)}')
        break
ml['drug_name'] = ml['drug_id'].map(name_map).fillna(ml['drug_id'])

In [ ]:
# ── 2. Holdout split (PRD: MDA-MB-231 + MCF7 + 20% TNBC unseen) ───────
import numpy as np, os
# NOTE: exclude the 'drug_id' STRING column — features are pca_* + drug_DPC-* dummies only
# NOTE: exclude BOTH string cols ('drug_id' AND 'drug_name' start with 'drug_'!) — features are pca_* + drug_DPC-* dummies only
FEAT = [c for c in ml.columns if c.startswith('pca_') or (c.startswith('drug_') and c not in ('drug_id', 'drug_name'))]
print(f'n_features = {len(FEAT)} (128 PCA + {len(FEAT)-128} drug dummies)')
X = ml[FEAT].to_numpy(dtype=np.float32)
y = ml['auc'].to_numpy(dtype=np.float32)
groups = ml['ModelID'].to_numpy()
print(f'X {X.shape} float32 = {X.nbytes/1e6:.0f} MB ✅ Colab-safe')

HOLDOUT_FIXED = {'ACH-000768', 'ACH-000019'}  # MDA-MB-231 (TNBC) + MCF7 (Luminal control)
assert HOLDOUT_FIXED.issubset(set(groups)), 'Holdout lines missing from dataset!'
_hp = OUT + '/holdout_lines.npy'
if os.path.exists(_hp):
    # SAME split 01 used for feature fitting (upload it alongside the parquet) ✅
    HOLDOUT = set(np.load(_hp, allow_pickle=True).tolist()) | HOLDOUT_FIXED
    print('holdout loaded from 01 §9a.')
else:  # fallback: identical logic (seed 42) — only if the .npy was not uploaded
    rng = np.random.RandomState(42)
    rest = sorted(set(groups) - HOLDOUT_FIXED)
    extra = set(rng.choice(rest, size=max(1, len(rest)//5), replace=False))
    HOLDOUT = HOLDOUT_FIXED | set(extra)
HOLDOUT = {h for h in HOLDOUT if h in set(groups)}
is_test = np.array([g in HOLDOUT for g in groups])
print(f'train {int((~is_test).sum()):,} | holdout {int(is_test.sum()):,} | holdout lines ({len(HOLDOUT)}): {sorted(HOLDOUT)[:5]}...')
print(json.dumps({'holdout_lines': sorted(HOLDOUT), 'seed': 42}, indent=1))
np.save(OUT + '/holdout_lines.npy', np.array(sorted(HOLDOUT)))

In [ ]:
# ── 3. Ranking metrics (PRD FR-06, primary for project) ───────────────
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

def dcg_at_k(rels, k=10):
    rels = np.asarray(rels)[:k]
    return float(np.sum((2**rels - 1) / np.log2(np.arange(2, len(rels)+2))))

def ndcg_at_k(y_true_auc, y_pred_auc, k=10):
    """Graded relevance = 1 - AUC (low AUC = high relevance). Lower AUC = more sensitive."""
    true_rel = 1.0 - np.asarray(y_true_auc)
    order_pred = np.argsort(y_pred_auc)          # predicted most-sensitive first
    order_ideal = np.argsort(y_true_auc)
    ideal = dcg_at_k(true_rel[order_ideal], k)
    if ideal == 0: return 0.0
    return dcg_at_k(true_rel[order_pred], k) / ideal

def recall_at_k(y_true_auc, y_pred_auc, k=10, thresh=0.3):
    """Sensitive = AUC < 0.3 (PRD). Recall = hits_in_topK / total_sensitive."""
    y_true_auc = np.asarray(y_true_auc); y_pred_auc = np.asarray(y_pred_auc)
    sensitive = set(np.where(y_true_auc < thresh)[0])
    if len(sensitive) == 0: return np.nan  # no sensitive drugs for this line → skip in mean
    topk = set(np.argsort(y_pred_auc)[:k])
    return len(sensitive & topk) / len(sensitive)

def per_line_spearman(df, true_col='auc', pred_col='pred'):
    vals = []
    for _, g in df.groupby('ModelID'):
        if len(g) > 5:
            vals.append(spearmanr(g[true_col], g[pred_col]).statistic)
    vals = [v for v in vals if not np.isnan(v)]
    return float(np.mean(vals)) if vals else float('nan'), vals

def report(y_true, y_pred, df_test, label):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    d = df_test.copy(); d['pred'] = y_pred
    mean_sp, _ = per_line_spearman(d)
    ndcgs, recalls = [], []
    for _, g in d.groupby('ModelID'):
        ndcgs.append(ndcg_at_k(g['auc'].values, g['pred'].values, 10))
        r = recall_at_k(g['auc'].values, g['pred'].values, 10, 0.3)
        if not np.isnan(r): recalls.append(r)
    print(f"{label:22s} MAE {mean_absolute_error(y_true, y_pred):.4f} | RMSE {rmse:.4f} | "
          f"R² {r2_score(y_true, y_pred):.4f} | Spearman_global {spearmanr(y_true, y_pred).statistic:.4f} | "
          f"per-line_Spearman {mean_sp:.4f} | NDCG@10 {np.mean(ndcgs):.4f} | Recall@10 {np.mean(recalls) if recalls else float('nan'):.4f}")
    return {'label': label, 'mae': float(mean_absolute_error(y_true, y_pred)), 'rmse': float(rmse),
            'r2': float(r2_score(y_true, y_pred)), 'spearman': float(spearmanr(y_true, y_pred).statistic),
            'per_line_spearman': mean_sp, 'ndcg10': float(np.mean(ndcgs)),
            'recall10': float(np.mean(recalls)) if recalls else None}

⚠️ **Cell below has an intentional safeguard:** the `ndcg_at_k` function above). The next cell self-tests it so copy-paste still runs.

In [ ]:
# ── 3b. Metric self-test (safe to re-run; uses ndcg_at_k from §3 above) ───────────
import numpy as np
print('metrics OK:', round(ndcg_at_k([0.1,0.5,0.9],[0.15,0.5,0.85]),3), '(expect ~1.0 for good ranking)')


In [ ]:
# ── 4. Train: Dummy → RF → XGBoost → CatBoost (PRD FR-05 order) ───────
# Colab CPU budget <30 min: RF-200 + XGB-600 + CatBoost-800 ≈ 10-20 min on ~140k rows.
import time, json
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

Xtr, ytr, gtr = X[~is_test], y[~is_test], groups[~is_test]
Xte, yte = X[is_test], y[is_test]
df_test = ml.loc[is_test, ['ModelID','drug_id','drug_name','auc']].reset_index(drop=True)
results = []

# 4a. DummyMean baseline (MUST beat this — PRD)
t0=time.time()
dummy = DummyRegressor(strategy='mean').fit(Xtr, ytr)
results.append(report(yte, dummy.predict(Xte), df_test, 'DummyMean'))
print(f'  ({time.time()-t0:.1f}s)')

# 4b. GroupKFold sanity on train (k=5 by cell_line, show 1 fold RF for speed)
gkf = GroupKFold(n_splits=5)
a, b = next(gkf.split(Xtr, ytr, gtr))
rf_cv = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42).fit(Xtr[a], ytr[a])
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
print(f"GroupKFold demo fold: R²={r2_score(ytr[b], rf_cv.predict(Xtr[b])):.3f} | Spearman={spearmanr(ytr[b], rf_cv.predict(Xtr[b])).statistic:.3f}")
del rf_cv; gc.collect()

In [ ]:
# ── 4c. RandomForest (full train) ─────────────────────────────────────
import time, gc
from sklearn.ensemble import RandomForestRegressor
t0=time.time()
rf = RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=2, n_jobs=-1, random_state=42)
rf.fit(Xtr, ytr)
results.append(report(yte, rf.predict(Xte), df_test, 'RandomForest-300'))
print(f'  ({time.time()-t0:.1f}s)')
import pickle
pickle.dump(rf, open(OUT + '/rf_tnbc_v1.pkl','wb'))
print('saved /content/rf_tnbc_v1.pkl')

In [ ]:
# ── 4d. XGBoost ───────────────────────────────────────────────────────
import time
from xgboost import XGBRegressor
t0=time.time()
# NOTE: early_stopping_rounds was removed from the XGBRegressor constructor in xgboost>=2 —
# it must go to fit() (or callbacks in v3). Version-proof below.
xgb = XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', n_jobs=-1,
                   random_state=42, eval_metric='rmse')
# honest early-stop: last GroupKFold fold as eval set
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(Xtr, ytr, gtr))
a, b = splits[-1]
try:
    from xgboost.callback import EarlyStopping
    xgb.fit(Xtr[a], ytr[a], eval_set=[(Xtr[b], ytr[b])],
            callbacks=[EarlyStopping(rounds=50, save_best=True)], verbose=50)
except Exception as e:
    print('callbacks API unavailable, plain fit:', str(e)[:120])
    xgb.fit(Xtr[a], ytr[a], verbose=50)
results.append(report(yte, xgb.predict(Xte), df_test, 'XGBoost-600'))
print(f'  ({time.time()-t0:.1f}s) best_iter={getattr(xgb, "best_iteration", "?")}')
xgb.save_model(OUT + '/xgb_tnbc_v1.json')
print('saved /content/xgb_tnbc_v1.json')

In [ ]:
# ── 4e. CatBoost (expected best for MVP, PRD FR-05) ────────────────────
import time
from catboost import CatBoostRegressor
t0=time.time()
cat = CatBoostRegressor(iterations=800, depth=6, learning_rate=0.05, l2_leaf_reg=3.0,
                        loss_function='RMSE', random_seed=42, verbose=100,
                        early_stopping_rounds=50, task_type='CPU')
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
a, b = list(gkf.split(Xtr, ytr, gtr))[-1]
cat.fit(Xtr[a], ytr[a], eval_set=(Xtr[b], ytr[b]), use_best_model=True)
results.append(report(yte, cat.predict(Xte), df_test, 'CatBoost-800 ✅'))
print(f'  ({time.time()-t0:.1f}s) best_iter={cat.best_iteration_}')
cat.save_model(OUT + '/catboost_tnbc_v1.cbm')
print('saved /content/catboost_tnbc_v1.cbm')

# Leaderboard + PRD gate (R²>0.25, per-line Spearman>0.4 on breast — confirm after audit)
import pandas as pd
# Leaderboard sorted by PRIMARY metric (NDCG@10 ranking — PRD FR-06), not R²
lb = pd.DataFrame(results).sort_values('ndcg10', ascending=False)
display(lb)
best = lb.iloc[0]
print(f"\nBEST: {best['label']} | NDCG@10={best['ndcg10']:.3f} | R²={best['r2']:.3f} (need >0.25) | per-line Spearman={best['per_line_spearman']:.3f} (need >0.4)")
lb.to_csv(OUT + '/leaderboard.csv', index=False)

In [ ]:
# ── 5. OUTPUT 1 — Sensitivity predictor: P(cell, drug) → AUC ───────────
# Lower AUC = more sensitive = more killing. Never "cure" language (PRD §5.2).
import pandas as pd, numpy as np, pickle
from catboost import CatBoostRegressor

pca_df = pd.read_parquet([p for p in [
    OUT + '/expression_pca128_breast.parquet',
    OUT + '/drive/MyDrive/TNBC_expression_pca128_breast.parquet'] if os.path.exists(p)][0])
print('PCA lines:', pca_df.shape)
# Defensive: one row per line (dup PCA rows break the len==1 assert below)
pca_df = pca_df.drop_duplicates(subset=['ModelID'], keep='first').reset_index(drop=True)
print('PCA unique lines:', pca_df.shape)

best_model = CatBoostRegressor().load_model(OUT + '/catboost_tnbc_v1.cbm')
FEAT_COLS = FEAT  # from §2
# Exact column positions — never assume get_dummies order == sorted() order
POS = {c: i for i, c in enumerate(FEAT_COLS)}
pca_cols = [c for c in FEAT_COLS if c.startswith('pca_')]
drug_ids_all = sorted(ml['drug_id'].unique().tolist())
missing = [f'drug_{d}' for d in drug_ids_all if f'drug_{d}' not in POS]
assert not missing, f'drug one-hot columns missing: {missing[:5]}'
print(f'model covers {len(drug_ids_all)} drugs (all one-hot columns verified ✅)')

def predict_sensitivity(model_id: str, drug_id: str) -> float:
    """Predicted AUC in [0,1] for one cell_line × drug. Lower = more sensitive."""
    row = pca_df.loc[pca_df['ModelID'] == model_id]
    assert len(row) == 1, f'Unknown/no PCA for {model_id}'
    col = f'drug_{drug_id}'
    assert col in POS, f'Unknown drug {drug_id}'
    x = np.zeros((1, len(FEAT_COLS)), dtype=np.float32)
    for c in pca_cols:
        x[0, POS[c]] = float(row[c].iloc[0])
    x[0, POS[col]] = 1.0
    return float(np.clip(best_model.predict(x)[0], 0, 1))

# Demo: positive control Doxorubicin (find its DPC id by name) + one prediction
dox_ids = [d for d in drug_ids_all if 'DOXORUBICIN' in str(name_map.get(d,'')).upper()] if name_map else []
print('Doxorubicin DPC ids:', dox_ids[:3] if dox_ids else '(name map missing — using raw DPC ids)')
print('demo MDA-MB-231 ×', drug_ids_all[0], '→ predicted_AUC =', round(predict_sensitivity('ACH-000768', drug_ids_all[0]), 4))
print('demo MCF7       ×', drug_ids_all[0], '→ predicted_AUC =', round(predict_sensitivity('ACH-000019', drug_ids_all[0]), 4))

In [ ]:
# ── 6. OUTPUT 2 — Drugs that KILL TNBC: ranked Top-20 for MDA-MB-231 ───
# Rank = ascending predicted_AUC. Selectivity vs MCF7 flags TNBC-preferential killers.
import pandas as pd, numpy as np

TNBC_ID, CTRL_ID = 'ACH-000768', 'ACH-000019'  # MDA-MB-231 vs MCF7
rows = []
for d in drug_ids_all:
    a_tnbc = predict_sensitivity(TNBC_ID, d)
    a_ctrl = predict_sensitivity(CTRL_ID, d)
    rows.append({'drug_id': d, 'drug_name': name_map.get(d, d) if name_map else d,
                 'pred_AUC_MDA231': a_tnbc, 'pred_AUC_MCF7': a_ctrl,
                 'selectivity_delta': a_ctrl - a_tnbc})  # >0 = kills TNBC more than control
rank = pd.DataFrame(rows).sort_values('pred_AUC_MDA231').reset_index(drop=True)
rank['rank'] = rank.index + 1
rank['sensitive_call'] = np.where(rank['pred_AUC_MDA231'] < 0.3, 'SENSITIVE (AUC<0.3)', 'resistant/intermediate')
rank['selective_call'] = np.where(rank['selectivity_delta'] > 0.05, 'TNBC-selective', 'non-selective')
display(rank.head(20))

rank.to_csv(OUT + '/mda_mb_231_full_ranking.csv', index=False)
rank.head(20).to_csv(OUT + '/mda_mb_231_top20_killers.csv', index=False)
print('saved /content/mda_mb_231_top20_killers.csv (Top-20) + full ranking')
print(f"killers (AUC<0.3): {(rank['pred_AUC_MDA231']<0.3).sum()} / {len(rank)} | TNBC-selective in Top-20: {((rank.head(20)['selectivity_delta'])>0.05).sum()}")

# Wet-lab shortlist (PRD §10): Top 3-5 AI + Doxorubicin/Paclitaxel control + 1 predicted-resistant (bottom row)
short = pd.concat([rank.head(5), rank.tail(1)])
if name_map:
    dox = rank[rank['drug_name'].str.contains('DOXORUBICIN|PACLITAXEL', case=False, na=False)].head(2)
    short = pd.concat([rank.head(5), dox, rank.tail(1)]).drop_duplicates()
short.to_csv(OUT + '/wetlab_shortlist.csv', index=False)
print('saved /content/wetlab_shortlist.csv — candidates for MTT/CellTiter-Glo 72h dose-response')
display(short)

In [ ]:
# ── 7. Explainability: SHAP global Top-20 genes + local (PRD FR-07) ────
import shap, numpy as np, pandas as pd, matplotlib.pyplot as plt

# SHAP on PCA features is abstract — also map back to top genes via PCA components_
# sampled background for speed (Colab-safe)
bg = Xtr[np.random.RandomState(0).choice(len(Xtr), size=min(200, len(Xtr)), replace=False)]
explainer = shap.TreeExplainer(xgb if 'xgb' in globals() else rf)
sv = explainer.shap_values(bg[:50])
print('SHAP values shape:', np.asarray(sv).shape)
shap.summary_plot(sv, bg[:50], feature_names=FEAT, max_display=20, show=False)
plt.tight_layout(); plt.savefig(OUT + '/shap_global_top20.png', dpi=150)
print('saved /content/shap_global_top20.png')
plt.show()

# Local: top-1 killer prediction for MDA-MB-231
top1_drug = rank.iloc[0]['drug_id']
x_top1 = np.zeros((1, len(FEAT)), dtype=np.float32)
x_top1[0, :len(pca_cols)] = pca_df.loc[pca_df['ModelID']=='ACH-000768', pca_cols].to_numpy(dtype=np.float32)
x_top1[0, len(pca_cols) + drug_ids_all.index(top1_drug)] = 1.0
sv1 = explainer.shap_values(x_top1)
print(f'local SHAP for MDA-MB-231 × {top1_drug} ({name_map.get(top1_drug, top1_drug) if name_map else top1_drug}): pred_AUC={rank.iloc[0]["pred_AUC_MDA231"]:.3f}')
shap.waterfall_plot(shap.Explanation(values=np.asarray(sv1).ravel(), base_values=float(explainer.expected_value), data=x_top1[0], feature_names=FEAT), max_display=15, show=False)
plt.tight_layout(); plt.savefig(OUT + '/shap_local_mda231_top1.png', dpi=150)
print('saved /content/shap_local_mda231_top1.png')

In [ ]:
# ── 8. Persist everything (model_version traceability, PRD NFR) ────────
import json, os, datetime
meta = {
  'model_version': 'catboost-v1.0',
  'dataset': ML_PATH,
  'dataset_version': 'DepMap public-26q3-526b.11 + REPURPOSING (see 01 notebook)',
  'features': f'{len(FEAT)} (pca_128 + drug_one_hot)',
  'holdout': sorted(HOLDOUT),
  'seed': 42,
  'leaderboard': results,
  'outputs': ['mda_mb_231_top20_killers.csv','mda_mb_231_full_ranking.csv','wetlab_shortlist.csv',
              'candidate_rankings.csv','audit_summary.json (from 01)',
              'catboost_tnbc_v1.cbm','xgb_tnbc_v1.json','rf_tnbc_v1.pkl','shap_global_top20.png'],
  'disclaimer': 'predicted sensitivity in cell model — NOT a clinical recommendation (PRD §5.2)',
  'date': datetime.date.today().isoformat()
}
json.dump(meta, open(OUT + '/model_info.json','w'), indent=1)
print(json.dumps(meta, indent=1)[:1500])

# PRD/MY_OUTLINE artifact name: candidate_rankings.csv == wet-lab shortlist
import shutil as _shutil
_shutil.copy(OUT + '/wetlab_shortlist.csv', OUT + '/candidate_rankings.csv')
print('saved /content/candidate_rankings.csv (alias of wetlab_shortlist)')

# PRD FR-06: TNBC-slice metrics on holdout (needs 01 §6 annotation file; skipped silently if absent)
try:
    _ann = pd.read_csv(OUT + '/breast_lines_annotated.csv')
    _tnbc = set(_ann.loc[_ann['coarse_subtype'] == 'TNBC', 'ModelID'])
    _d = df_test.copy(); _d['pred'] = cat.predict(Xte)
    _d = _d[_d['ModelID'].isin(_tnbc)]
    _m, _ = per_line_spearman(_d)
    print(f'TNBC-slice holdout: {len(_d)} rows, {len(_tnbc & set(df_test["ModelID"]))} lines | per-line Spearman {_m:.4f} (need >0.4)')
except Exception as e:
    print('TNBC slice skipped (upload breast_lines_annotated.csv from 01 to enable):', str(e)[:120])

!if [ -d "/content/drive/MyDrive" ]; then cp /content/catboost_tnbc_v1.cbm /content/mda_mb_231_top20_killers.csv /content/mda_mb_231_full_ranking.csv /content/wetlab_shortlist.csv /content/candidate_rankings.csv /content/model_info.json /content/leaderboard.csv "/content/drive/MyDrive/" && echo drive-backup-ok; else echo no-drive-skip-all-files-stay-in-/content; fi
!ls -lh "{OUT}"/*.csv "{OUT}"/*.cbm "{OUT}"/*.json "{OUT}"/*.png 2>/dev/null | head -20

print('\n✅ DONE — DoD: model predicts AUC (Output 1, §5) + Top-20 TNBC killers ranked (Output 2, §6).')
print('Next: FastAPI /rank + Streamlit (PRD FR-08) → wet-lab dose-response → predicted-vs-experimental scatter (FR-09).')

## Reading the killer list (for thesis)

- **`pred_AUC_MDA231`** — predicted sensitivity. **Lower = stronger killing.** `<0.3` = sensitive call (PRD FR-06).
- **`selectivity_delta = MCF7 − MDA231`** — `>0.05` = kills TNBC preferentially (in-silico proxy for PRD §10 `IC50_MCF7/IC50_MDA231 > 1.5`; confirm in lab).
- **Wet-lab order:** Top 3–5 `SENSITIVE + TNBC-selective` rows + Doxorubicin/Paclitaxel positive control + 1 bottom-ranked resistant line from `wetlab_shortlist.csv`.
- **Success (PRD §10):** `Spearman(predicted_AUC, experimental_AUC) > 0` AND ≥1 candidate selective in vitro. Report as research signal, never as treatment advice.